# Compare Different Models on Real-World Datasets

Evaluate all registered models on OpenML benchmark datasets, with per-model cache reuse and W&B logging.

In [ ]:
from pathlib import Path
from typing import Any

from pfns.run_evaluation_cli import (
    build_available_baseline_model_configs,
    compute_per_dataset_stats,
    print_results_summary,
    run_real_world_model_from_config,
    summarize_results,
)
from pfns.experiments.model_benchmarks.hashing import single_model_hash
from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.io import (
    REAL_WORLD_BUNDLE_KEYS,
    REAL_WORLD_REQUIRED_FILES,
    download_results_bundle_from_wandb,
    find_latest_real_world_bundle_for_model,
    load_dataframe_bundle,
    make_bundle_path,
    make_model_artifact_name,
    
    sanitize_wandb_artifact_component,
    save_dataframe_bundle,
    upload_results_bundle_to_wandb,
)
from pfns.experiments.model_benchmarks.model_registry import (
    MODEL_FAMILIES,
    get_all_models,
    get_baseline_models,
    get_models_from_families,
    get_models_from_names,
)
from pfns.utils import build_exp_outputs_path
from pfns.experiments.model_benchmarks.real_world_presets import (
    get_real_world_preset,
)
from pfns.experiments.model_benchmarks.plotting import make_display_name_formatter
from pfns.experiments.model_benchmarks.workflows import (
    aggregate_real_world_results_from_bundles,
    alias_real_world_dataframe_bundle,
    build_real_world_run_metadata,
    load_cached_real_world_results_for_preset,
    real_world_bundle_is_compatible,
)

REAL_WORLD_PRESET = "openml" # "openml"  # or "tabarena"
preset_defaults = get_real_world_preset(REAL_WORLD_PRESET)
REAL_DATASET_EXPERIMENT: dict[str, Any] = preset_defaults["experiment"]

BASELINE_EVAL: dict[str, int] = {
    "n_jobs": 4,
    "random_state": 42,
}

WANDB: dict[str, Any] = {
    "enabled": True,
    "overwrite": False,
    "artifact_name_real_eval": "real_eval_results",
    "entity": "icl_arch",
    "mode": "online",
} | preset_defaults["wandb"]

format_model_label = make_display_name_formatter()

OUTPUT_ROOT = build_exp_outputs_path(Path.cwd(), "real_world_eval")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Results are stored in: {OUTPUT_ROOT}")
print(f"Available model families: {list(MODEL_FAMILIES)}")
print(f"Preset: {REAL_WORLD_PRESET}")


def get_model_combinations(*model_dicts, add_baselines=False):
    combined = {}
    if add_baselines:
        baseline_models_to_compare, skipped_baselines = build_available_baseline_model_configs(
            candidates=get_baseline_models(),
            n_jobs=BASELINE_EVAL["n_jobs"],
            random_state=BASELINE_EVAL["random_state"],
        )
        if skipped_baselines:
            print(f"Skipping unavailable baselines in this environment: {skipped_baselines}")
        combined.update(baseline_models_to_compare)
        
    if not model_dicts:
        return {}
    
    for d in model_dicts:
        combined.update(d)
    return combined

# Model selection

# All models
# models_to_compare = get_model_combinations(get_all_models(), add_baselines=True)

# All models without baselines
# models_to_compare = get_model_combinations(get_all_models(), add_baselines=False)

# Only baselines
# models_to_compare = get_model_combinations( {}, add_baselines=True)

# Equal param models
# models_to_compare = get_model_combinations(get_models_from_families(["equal_params"]), add_baselines=True)

# Transformer masking expreiments
# models_to_compare = get_model_combinations(get_models_from_families(["transformer_masked"]), add_baselines=False)

# All FLA models
# models_to_compare = get_model_combinations(get_models_from_families(["fla_models"]), add_baselines=False)

# models_to_compare =  get_model_combinations(get_models_from_names([
#     "Bidirectional_DeltaNet_Comb_ST", 
#     "DeltaNet_Comb_ST_Reference"
#     ]
# ), add_baselines=False)

# models_to_compare = get_models_from_families(["bidirectional"])
# models_to_compare.update(get_models_from_names(["Linear_Attention_Non_Causal", "Linear_Attention_FLA_Comb_ST", "GLA_Comb_ST"]))

# models_to_compare = get_models_from_names([
#     "Softmax_Transformer",
#     "masked:Transformer_Comb_ST",
#     # "Softmax_Transformer_Cat_10_Training",
#     # "Softmax_Transformer_No_Cat_Norm",
#     # "Softmax_Transformer_Full_Cat_Norm",
#     # "equal_params:Transformer_Comb_ST",
#     #"Rebased_feat_dim_80",   (currently broken on tabarena, needs reevaluation on larger gpu)
#     # "Rebased_feat_dim_32",
#     # "equal_params:Rebased_Comb_ST"
#     "Linear_Attention_Non_Causal",
#     "Linear_Attention_Comb_ST",
#     # "Linear_Attention_Non_Causal_updated",
#     # "Linear_Attention_Comb_ST_updated",
#     # "Linear_Attention_Comb_ST_old_setup",
#     # "Linear_Attention_FLA_Comb_ST",
#     # "equal_params:Linear_Attention_Comb_ST",
#     # "equal_params:Linear_Attention_FLA_Comb_ST",
#     # "Linear_Attention_FLA_Comb_ST_wo_self_term",
#     # "Linear_Attention_Non_Causal_with_k_sum_norm"
#     #"equal_params:DeltaNet_Comb_ST"
# ])

# models_to_compare = get_model_combinations(get_models_from_families(["deltanet_high_seq_len"]), add_baselines=False)
models_to_compare = get_model_combinations(get_models_from_families(["equal_params_new"]), add_baselines=False)

# models_to_compare =  get_model_combinations(get_models_from_names([
#     "Transformer_Non_Causal_new", 
#     ]
# ), add_baselines=False)

# models_to_compare = get_models_from_names([
#     # "State_Passing_GLA_Comb_ST",
#     # "State_Passing_GLA_Comb_ST_new",
#     # "State_Passing_DeltaNet_Comb_ST",
#     # "State_Weaving_DeltaNet_Comb_ST",
#     # "mimetic:GLA_Comb_ST_Ref",
#     # "mimetic:GLA_Comb_ST_mimetic_gate_only",
#     # "Bidirectional_DeltaNet_Comb_ST_mean_output_mean_cache",
#     # "Bidirectional_GLA_Comb_ST_mean_output_mean_cache",
#     "DeltaNet_Comb_ST_Reference_New",
#     # "DeltaNet_Comb_ST_Seq_Len_200-64K_lognormal_dataset_matched",
#     # "subsampled:GLA_Comb_ST_3K",
#     #"Decay_LR_DeltaNet_Comb_ST",
#     "Linear_Attention_Non_Causal",
#     # "Linear_Attention_Comb_ST",
#     "Softmax_Transformer",
# ])


print(
    "Selected models:", len(models_to_compare)
)

device = get_default_device()
print(f"Using device: {device}")

expected_real_metadata = build_real_world_run_metadata(
    experiment=REAL_DATASET_EXPERIMENT,
    device=device,
)


In [ ]:
if WANDB["enabled"] and WANDB["overwrite"]:
    print("WANDB overwrite=True: skipping per-model download and forcing reruns.")

for model_name, model_config in models_to_compare.items():
    model_hash = single_model_hash(
        model_name=model_name,
        model_config=model_config,
        experiment_payload=REAL_DATASET_EXPERIMENT,
    )
    model_artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name_real_eval"],
        model_name=model_name,
        model_hash=model_hash,
    )

    if WANDB["enabled"] and not WANDB["overwrite"]:
        cached_bundle_path = download_results_bundle_from_wandb(
            artifact_name=model_artifact_name,
            download_root=OUTPUT_ROOT / "wandb_model_cache" / "real_world",
            required_files=REAL_WORLD_REQUIRED_FILES,
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
        )
        print(f"Checked for cached W&B artifact for {model_name}: {cached_bundle_path}")
        if cached_bundle_path is not None:
            cached_bundle = load_dataframe_bundle(
                cached_bundle_path,
                expected_keys=REAL_WORLD_BUNDLE_KEYS,
            )
            cached_bundle_for_model, source_labels = alias_real_world_dataframe_bundle(
                cached_bundle,
                target_model_name=model_name,
            )
            cached_dataframes = cached_bundle_for_model["dataframes"]

            if real_world_bundle_is_compatible(
                cached_bundle_for_model,
                model_name=model_name,
                expected_metadata=expected_real_metadata,
            ):
                model_bundle_path = make_bundle_path(
                    OUTPUT_ROOT / "real_world",
                    f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
                )
                save_dataframe_bundle(
                    dataframes=cached_dataframes,
                    bundle_dir=model_bundle_path,
                    experiment=REAL_DATASET_EXPERIMENT,
                    run_metadata=expected_real_metadata,
                )
                if source_labels:
                    print(
                        f"Reused cached real-world W&B result for {model_name} from stored labels "
                        f"{sorted(source_labels)}: {cached_bundle_path}. Saved local alias bundle: {model_bundle_path}"
                    )
                else:
                    print(
                        f"Reused cached real-world W&B result for {model_name}: {cached_bundle_path}. "
                        f"Saved local alias bundle: {model_bundle_path}"
                    )
                continue

    print(f"Running real-world benchmark for model: {model_name}")
    results = run_real_world_model_from_config(
        model_config=model_config,
        experiment=REAL_DATASET_EXPERIMENT,
        device=device,
        baseline_n_jobs=BASELINE_EVAL["n_jobs"],
        random_state=BASELINE_EVAL["random_state"],
        verbose=False,
    )

    if not results.empty:
        results["model"] = model_name
    else:
        print(f"Warning: No results for model {model_name}, skipping saving and upload.")
        continue
    summary = summarize_results(results)
    per_dataset = compute_per_dataset_stats(results)

    model_bundle_path = make_bundle_path(
        OUTPUT_ROOT / "real_world",
        f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_dataframe_bundle(
        dataframes={
            "results": results,
            "summary": summary.reset_index() if summary is not None else None,
            "per_dataset": per_dataset,
        },
        bundle_dir=model_bundle_path,
        experiment=REAL_DATASET_EXPERIMENT,
        run_metadata=expected_real_metadata,
    )
    print(f"Saved real-world bundle for {model_name}: {model_bundle_path}")

    if WANDB["enabled"]:
        artifact_ref = upload_results_bundle_to_wandb(
            model_bundle_path,
            artifact_name=model_artifact_name,
            run_name=f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}_{model_hash}_artifact",
            metadata={
                "experiment": REAL_DATASET_EXPERIMENT,
                "model_name": model_name,
                "model_config": model_config,
                "model_hash": model_hash,
                "run_metadata": expected_real_metadata,
            },
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
            job_type="real_world_bundle_upload",
        )
        print(f"Uploaded real-world artifact for {model_name}: {artifact_ref}")

print("\nReal-world evaluation completed.")


## Load and aggregate per-model outputs

Collect the latest compatible bundle for each selected model, then build combined result tables.

In [ ]:
from IPython.display import display

from pfns.experiments.model_benchmarks.real_world_benchmarks import (
    get_real_world_benchmark_dataset_count,
)

bundles_by_model = {}
missing_models = []

for model_name in models_to_compare:
    bundle_path, bundle = find_latest_real_world_bundle_for_model(
        model_name,
        output_root=OUTPUT_ROOT,
        expected_metadata=expected_real_metadata,
    )
    if bundle is None:
        missing_models.append(model_name)
        continue

    bundles_by_model[model_name] = {"path": bundle_path, "bundle": bundle}
    print(f"Loaded {model_name} bundle: {bundle_path}")

if not bundles_by_model:
    raise RuntimeError(
        "No compatible bundles were found. Run the benchmark cell first, or check metadata settings."
    )

all_results, results_by_model = aggregate_real_world_results_from_bundles(
    bundles_by_model,
    expected_splits=int(REAL_DATASET_EXPERIMENT["n_splits"]),
)

summary = summarize_results(all_results)
per_dataset = compute_per_dataset_stats(all_results)
if summary is None or per_dataset is None or per_dataset.empty:
    raise RuntimeError("Could not compute summary tables from aggregated results.")

expected_dataset_count = get_real_world_benchmark_dataset_count(
    REAL_DATASET_EXPERIMENT["benchmark"]
)

print("\nAggregated real-world benchmark summary")
print(f"Models loaded: {len(bundles_by_model)} / {len(models_to_compare)}")
print(f"Rows in all_results: {len(all_results)}")
print(f"Datasets covered: {per_dataset['dataset'].nunique()} / {expected_dataset_count}")

if missing_models:
    print("Missing compatible bundles for:", missing_models)

summary_metrics = [
    {"metric": "accuracy", "show_std": True},
    {"metric": "roc_auc", "show_std": True},
    #{"metric": "log_loss", "show_std": True},
    #{"metric": "ece", "show_std": True},
    {"metric": "fit_time", "show_std": False},
    {"metric": "predict_time", "show_std": False},
]

print_results_summary(
    all_results,
    title="Aggregated Results Across Selected Models",
    metrics=summary_metrics,
)

## Combined OpenML and TabArena table

Display OpenML CC-18 and TabArena model summaries in one multi-column table.

In [ ]:
import pandas as pd

from pfns.experiments.model_benchmarks.plotting import resolve_display_name_map
from pfns.experiments.model_benchmarks.latex_tables import (
    apply_real_world_latex_cell_formatting,
    latex_escape,
    make_real_world_metric_tables,
    make_real_world_significance_table,
    render_combined_real_world_latex_table,
    sort_models_by_reference_metric,
)

COMBINED_REAL_WORLD_TABLES = {
    "openml": r"\shortstack{OpenML CC-18\\(filtered \& subsampled)}",
    "tabarena": r"\shortstack{TabArena\\(feature subsampled ($\le 20$))}",
}
COMBINED_REAL_WORLD_METRICS = {
    "roc_auc": r"ROC-AUC $\uparrow$",
    "accuracy": r"ACC $\uparrow$",
    "log_loss": r"CE $\downarrow$",
}
COMBINED_REAL_WORLD_ADD_SIGNIFICANCE_MARKERS = True
COMBINED_REAL_WORLD_SIGNIFICANCE_ALPHA = 0.05
COMBINED_REAL_WORLD_BASELINE_MODEL_NAMES = set(get_baseline_models())
COMBINED_REAL_WORLD_UNRANKED_MODEL_NAMES = {
#     "equal_params:Transformer_Comb_ST",
#     "equal_params:Rebased_feat_dim_32",
#     "Linear_Attention_Non_Causal",
}
COMBINED_REAL_WORLD_DISPLAY_NAME_OVERRIDES = {
    "Non-Causal Transformer equal params": "Non-Causal Transformer",
    "Rebased $\\phi$ with 32-dim features": "Rebased w. 32-dim features",
    "Linear Attention (FLA; Comb ST)": "Linear Attention (FLA)",
    "Linear Attention (Comb_ST)": "Linear Attention",
}
COMBINED_REAL_WORLD_CAPTION = (
    "Model comparison on OpenML CC-18 and TabArena. OpenML CC-18 uses the "
    "30-dataset subset from TabPFN v1 \\cite{hollmann2023tabpfn}, with samples "
    "and features subsampled to at most 1000 and 20. TabArena uses its 38 "
    "classification datasets with at most 20 features. We report means over "
    "5-fold stratified cross-validation, aggregating splits and datasets.  "
    "All models use the Combined Single Target training strategy. Within the "
    "causal linear recurrence block, the best model per metric is underlined. "
    "The best reference baseline is additionally marked in bold."
)
COMBINED_REAL_WORLD_SIGNIFICANCE_CAPTION = (
    " Superscript letters denote pairwise significance groups within each benchmark "
    "metric. Models sharing at least one letter are not significantly different. "
    "Significance is computed over dataset-level raw scores using paired two-sided "
    "Wilcoxon signed-rank tests with Holm's alpha (5\\%) correction."
)


def format_combined_real_world_model_name(model):
    display_name = str(combined_display_name_map.get(str(model), str(model))).replace("\n", " ")
    return latex_escape(COMBINED_REAL_WORLD_DISPLAY_NAME_OVERRIDES.get(display_name, display_name))


combined_real_world_main_tables = {}
combined_real_world_significance_tables = {}
combined_missing_by_benchmark = {}
combined_display_name_map = {}
for preset_name, benchmark_label in COMBINED_REAL_WORLD_TABLES.items():
    preset_results, preset_missing = load_cached_real_world_results_for_preset(
        preset_name,
        model_names=models_to_compare,
        device=device,
        output_root=OUTPUT_ROOT,
    )
    combined_missing_by_benchmark[benchmark_label] = preset_missing
    if preset_results.empty:
        print(f"No compatible cached bundles found for {benchmark_label}.")
        continue

    combined_display_name_map.update(resolve_display_name_map(preset_results))
    main_table, per_dataset_stats = make_real_world_metric_tables(
        preset_results,
        metric_labels=COMBINED_REAL_WORLD_METRICS,
        compute_per_dataset_stats=compute_per_dataset_stats,
    )
    combined_real_world_main_tables[benchmark_label] = main_table
    ranked_models = [
        model
        for model in main_table.index
        if model not in COMBINED_REAL_WORLD_BASELINE_MODEL_NAMES
        and model not in COMBINED_REAL_WORLD_UNRANKED_MODEL_NAMES
    ]
    if COMBINED_REAL_WORLD_ADD_SIGNIFICANCE_MARKERS:
        combined_real_world_significance_tables[benchmark_label] = make_real_world_significance_table(
            per_dataset_stats,
            metric_labels=COMBINED_REAL_WORLD_METRICS,
            benchmark_label=benchmark_label,
            target_models=ranked_models,
            output_models=list(main_table.index),
            alpha=COMBINED_REAL_WORLD_SIGNIFICANCE_ALPHA,
        )

if not combined_real_world_main_tables:
    raise RuntimeError("No compatible cached bundles were found for the combined OpenML/TabArena table.")

combined_real_world_main_table = pd.concat(combined_real_world_main_tables, axis=1)
combined_real_world_significance_table = (
    pd.concat(combined_real_world_significance_tables.values(), axis=1)
    if combined_real_world_significance_tables
    else None
)
combined_model_order = sort_models_by_reference_metric(
    combined_real_world_main_table,
    benchmark_label=COMBINED_REAL_WORLD_TABLES["openml"],
    metric_label=COMBINED_REAL_WORLD_METRICS["roc_auc"],
    baseline_model_names=COMBINED_REAL_WORLD_BASELINE_MODEL_NAMES,
    unranked_model_names=COMBINED_REAL_WORLD_UNRANKED_MODEL_NAMES,
)
combined_real_world_main_table = combined_real_world_main_table.reindex(combined_model_order)
if combined_real_world_significance_table is not None:
    combined_real_world_significance_table = combined_real_world_significance_table.reindex(combined_model_order)

combined_model_groups = {
    "ranked": [
        model
        for model in combined_model_order
        if model not in COMBINED_REAL_WORLD_BASELINE_MODEL_NAMES
        and model not in COMBINED_REAL_WORLD_UNRANKED_MODEL_NAMES
    ],
    "unranked": [model for model in combined_model_order if model in COMBINED_REAL_WORLD_UNRANKED_MODEL_NAMES],
    "baseline": [model for model in combined_model_order if model in COMBINED_REAL_WORLD_BASELINE_MODEL_NAMES],
}
for benchmark_label, preset_missing in combined_missing_by_benchmark.items():
    if preset_missing:
        print(f"Missing compatible bundles for {benchmark_label}: {preset_missing}")

combined_real_world_table = apply_real_world_latex_cell_formatting(
    combined_real_world_main_table,
    significance_table=combined_real_world_significance_table,
    baseline_model_names=COMBINED_REAL_WORLD_BASELINE_MODEL_NAMES,
    unranked_model_names=COMBINED_REAL_WORLD_UNRANKED_MODEL_NAMES,
).rename(index=format_combined_real_world_model_name)
combined_real_world_latex = render_combined_real_world_latex_table(
    combined_real_world_table,
    caption=COMBINED_REAL_WORLD_CAPTION
    + (COMBINED_REAL_WORLD_SIGNIFICANCE_CAPTION if COMBINED_REAL_WORLD_ADD_SIGNIFICANCE_MARKERS else ""),
    label="tab:openml_tabarena_summary",
    row_counts=[len(combined_model_groups[key]) for key in ("ranked", "unranked", "baseline")],
    tabcolsep="4pt",
    arraystretch="1.0",
)
print(combined_real_world_latex)


## Result tables

Display model-level and dataset-level metric tables from the aggregated benchmark results.

In [ ]:
from pfns.experiments.model_benchmarks.analysis import add_normalized_comparison_metrics
from pfns.experiments.model_benchmarks.plotting import (
    make_display_name_formatter,
    resolve_display_name_map,
)

if summary is None or per_dataset is None or summary.empty or per_dataset.empty:
    raise RuntimeError("No summary/per-dataset results available.")

display_name_map = resolve_display_name_map(all_results)
format_model_label = make_display_name_formatter(display_name_map=display_name_map)

summary_display = summary.copy().sort_values("roc_auc_mean", ascending=False)
summary_display.index = summary_display.index.map(format_model_label)
summary_metric_labels = {
    "accuracy_mean": "Accuracy mean \u2191",
    "accuracy_std": "Accuracy std \u2191",
    "roc_auc_mean": "ROC-AUC mean \u2191",
    "roc_auc_std": "ROC-AUC std \u2191",
    "log_loss_mean": "CE mean \u2193",
    "log_loss_std": "CE std \u2193",
    "ece_mean": "ECE mean \u2193",
    "ece_std": "ECE std \u2193",
}
summary_display = summary_display.rename(columns=summary_metric_labels)
print("Model summary table (mean over datasets):")
display(
    summary_display.style.format(
        {
            "Accuracy mean \u2191": "{:.4f}",
            "Accuracy std \u2191": "{:.4f}",
            "ROC-AUC mean \u2191": "{:.4f}",
            "ROC-AUC std \u2191": "{:.4f}",
            "CE mean \u2193": "{:.4f}",
            "CE std \u2193": "{:.4f}",
            "ECE mean \u2193": "{:.4f}",
            "ECE std \u2193": "{:.4f}",
            "fit_time_mean": "{:.3f}",
            "predict_time_mean": "{:.3f}",
        }
    ).background_gradient(cmap="Greens_r", subset=["Accuracy mean \u2191", "ROC-AUC mean \u2191"])
)

per_dataset_metric_specs = [
    ("accuracy_mean", "Accuracy \u2191"),
    ("roc_auc_mean", "ROC-AUC \u2191"),
    ("log_loss_mean", "CE \u2193"),
    ("ece_mean", "ECE \u2193"),
]
for metric, metric_label in per_dataset_metric_specs:
    print(f"\nPer-dataset table: {metric_label}")
    table = (
        per_dataset
        .pivot_table(index="dataset", columns="model", values=metric, observed=True)
        .sort_index()
    )
    table = table.rename(columns=format_model_label)
    display(table.style.format("{:.4f}").background_gradient(cmap="Blues"))

normalized_table_specs = [
    ("accuracy_mean", "normalized_accuracy_mean", "Normalized Accuracy ↑"),
    ("roc_auc_mean", "normalized_roc_auc_mean", "Normalized ROC-AUC ↑"),
    ("log_loss_mean", "normalized_log_loss_mean", "Normalized CE ↑"),
]
available_normalized_table_specs = [
    spec for spec in normalized_table_specs if spec[0] in per_dataset.columns
]
if not available_normalized_table_specs:
    raise RuntimeError("No supported per-dataset metric columns are available for normalization.")

raw_normalized_metric_cols = [spec[0] for spec in available_normalized_table_specs]
normalized_metric_df = add_normalized_comparison_metrics(
    per_dataset.melt(
        id_vars=["model", "dataset"],
        value_vars=raw_normalized_metric_cols,
        var_name="metric",
        value_name="value",
    ).dropna(subset=["value"]),
    metric_keys=raw_normalized_metric_cols,
    higher_is_better_metrics={"accuracy_mean", "roc_auc_mean"},
    group_cols=("dataset",),
)
normalized_metric_df = normalized_metric_df[
    normalized_metric_df["metric"].astype(str).str.startswith("normalized_")
].copy()
if normalized_metric_df.empty:
    raise RuntimeError("No normalized per-dataset metrics could be computed.")

normalized_label_map = {
    normalized_metric: metric_label
    for _, normalized_metric, metric_label in available_normalized_table_specs
}
normalized_summary_display = (
    normalized_metric_df.groupby(["model", "metric"], observed=True)["value"]
    .mean()
    .reset_index()
    .pivot_table(index="model", columns="metric", values="value", observed=True)
    .reindex(index=summary.index.tolist())
)
normalized_summary_display = normalized_summary_display.rename(columns=normalized_label_map)
normalized_summary_display = normalized_summary_display.sort_values(
    "Normalized ROC-AUC ↑",
    ascending=False,
)
normalized_summary_display.index = normalized_summary_display.index.map(format_model_label)
print("\nNormalized model score table (mean over datasets, higher is better):")
display(
    normalized_summary_display.style.format("{:.4f}").background_gradient(
        cmap="Greens",
        vmin=0.0,
        vmax=1.0,
        axis=None,
    )
)

for _, normalized_metric, metric_label in available_normalized_table_specs:
    print(f"\nNormalized per-dataset table: {metric_label}")
    table = (
        normalized_metric_df[normalized_metric_df["metric"] == normalized_metric]
        .pivot_table(index="dataset", columns="model", values="value", observed=True)
        .sort_index()
        .rename(columns=format_model_label)
    )
    display(
        table.style.format("{:.4f}").background_gradient(
            cmap="Blues",
            vmin=0.0,
            vmax=1.0,
            axis=None,
        )
    )



## Plots

Create metric bars, speed-vs-accuracy tradeoff, and dataset-level rank curves using existing benchmark utilities.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

from pfns.experiments.model_benchmarks.analysis import compute_mean_rank_tables
from pfns.experiments.model_benchmarks.constants import DEFAULT_COLORS
from pfns.experiments.model_benchmarks.plotting import (
    build_model_style_map,
    plot_curves_from_df,
    resolve_display_name_map,
)

if summary is None or summary.empty:
    raise RuntimeError("No summary dataframe available for plotting.")

display_name_map = resolve_display_name_map(all_results)

ordered_models = summary.sort_values("accuracy_mean", ascending=False).index.tolist()
non_black_colors = [c for c in DEFAULT_COLORS if c.lower() not in {"#000000", "black", "k"}]
style_map = build_model_style_map(ordered_models, colors=non_black_colors)
model_colors = {name: style_map[name][2] for name in ordered_models}
metric_labels = {
    "accuracy": "Accuracy \u2191",
    "roc_auc": "ROC-AUC \u2191",
    "log_loss": "CE \u2193",
}

SUMMARY_TITLE_FONTSIZE = 20
SUMMARY_XTICK_FONTSIZE = 15
SUMMARY_YTICK_FONTSIZE = 15

SUMMARY_ERROR_BAR_MODE = None # "std"  # 'std' or None
error_bar_title_suffix = " (std)" if SUMMARY_ERROR_BAR_MODE == "std" else ""
error_col_by_mode = {
    "std": {
        "accuracy": "accuracy_std",
        "roc_auc": "roc_auc_std",
        "log_loss": "log_loss_std",
    },
    None: {
        "accuracy": None,
        "roc_auc": None,
        "log_loss": None,
    },
}
if SUMMARY_ERROR_BAR_MODE not in error_col_by_mode:
    raise ValueError("SUMMARY_ERROR_BAR_MODE must be 'std' or None.")

# 1) Summary metric bar plots
bar_specs = [
    ("accuracy", "accuracy_mean", metric_labels["accuracy"], True),
    ("roc_auc", "roc_auc_mean", metric_labels["roc_auc"], True),
    ("log_loss", "log_loss_mean", metric_labels["log_loss"], False),
]

fig = plt.figure(figsize=(20, 12.5), dpi=400)
grid = fig.add_gridspec(2, 10, hspace=0.20, wspace=0.40)
axes = [
    fig.add_subplot(grid[0, :4]),
    fig.add_subplot(grid[0, 6:]),
    fig.add_subplot(grid[1, 3:7]),
]
for ax, (metric_key, mean_col, title, higher_is_better) in zip(axes, bar_specs):
    plot_df = summary.reset_index().rename(columns={"index": "model"})
    plot_df = plot_df.sort_values(mean_col, ascending=not higher_is_better)
    plot_df["display_model"] = plot_df["model"].map(lambda model: display_name_map.get(str(model), str(model)))

    err_col = error_col_by_mode[SUMMARY_ERROR_BAR_MODE][metric_key]
    err = plot_df[err_col].fillna(0.0) if err_col is not None else None
    ax.barh(
        plot_df["display_model"],
        plot_df[mean_col],
        xerr=err,
        color=[model_colors[m] for m in plot_df["model"]],
        alpha=0.9,
    )

    lower_series = plot_df[mean_col] - err if err is not None else plot_df[mean_col]
    upper_series = plot_df[mean_col] + err if err is not None else plot_df[mean_col]
    lower = float(lower_series.min())
    upper = float(upper_series.max())
    span = max(upper - lower, 1e-6)
    pad = 0.08 * span
    x_min = lower - pad
    x_max = upper + pad

    if mean_col in {"accuracy_mean", "roc_auc_mean"}:
        x_min = max(0.0, x_min)
        x_max = min(1.0, x_max)
    else:
        x_min = max(0.0, x_min)

    if x_max <= x_min:
        x_max = x_min + max(span, 1e-3)

    ax.set_xlim(x_min, x_max)
    
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
    ax.tick_params(axis="x", labelsize=SUMMARY_XTICK_FONTSIZE)
    
    ax.set_title(f"{title}{error_bar_title_suffix}", fontsize=SUMMARY_TITLE_FONTSIZE, pad=10)
    ax.grid(axis="x", alpha=0.25)
    ax.tick_params(axis="y", labelsize=SUMMARY_YTICK_FONTSIZE, labelleft=True, labelright=False, pad=4)

fig.subplots_adjust(left=0.20, right=0.98, top=0.93, bottom=0.08)
plt.show()

# 2) Accuracy vs inference speed tradeoff
tradeoff_df = summary.reset_index().rename(columns={"index": "model"})
tradeoff_df["display_model"] = tradeoff_df["model"].map(
    lambda model: display_name_map.get(str(model), str(model))
)
fig, ax = plt.subplots(figsize=(9, 6), dpi=400)
for _, row in tradeoff_df.iterrows():
    model_name = row["model"]
    display_model_name = row["display_model"]
    ax.scatter(
        row["predict_time_mean"],
        row["accuracy_mean"],
        s=90,
        color=model_colors[model_name],
        alpha=0.9,
        label=display_model_name,
    )
    ax.annotate(display_model_name, (row["predict_time_mean"], row["accuracy_mean"]), xytext=(4, 4), textcoords="offset points", fontsize=9)

ax.set_xlabel("Predict time mean (s)")
ax.set_ylabel(f"{metric_labels['accuracy']} mean")
ax.set_xscale("log")
ax.grid(alpha=0.25)
ax.set_title(f"{metric_labels['accuracy']} vs Prediction Time (s) \u2193")
plt.show()

# 3) Dataset-wise rank curves with model_benchmarks rank + plotting helpers
rank_base = (
    all_results
    .groupby(["model", "dataset"], observed=True)
    .agg(
        accuracy=("accuracy", "mean"),
        roc_auc=("roc_auc", "mean"),
        log_loss=("log_loss", "mean"),
    )
    .reset_index()
)

rank_long = rank_base.melt(
    id_vars=["model", "dataset"],
    value_vars=["accuracy", "roc_auc", "log_loss"],
    var_name="metric",
    value_name="value",
)
rank_long["metric"] = rank_long["metric"].replace({"accuracy": "acc"})

dataset_to_idx = {
    name: idx
    for idx, name in enumerate(sorted(rank_long["dataset"].unique()), start=1)
}
rank_input = rank_long.assign(seqlen=rank_long["dataset"].map(dataset_to_idx))
rank_input["rep"] = rank_input["dataset"]

rank_tables = compute_mean_rank_tables(
    rank_input,
    x_col="seqlen",
    higher_is_better_metrics={"acc", "roc_auc"},
)

overall_ranks = rank_tables["overall_ranks"].copy()
overall_ranks["metric"] = overall_ranks["metric"].replace({"acc": "accuracy"})
rank_metric_labels = {
    "accuracy": "Accuracy Rank \u2193",
    "roc_auc": "ROC-AUC Rank \u2193",
    "log_loss": "CE Rank \u2193",
}
print("Overall mean rank (lower is better \u2193):")
for metric in ["accuracy", "roc_auc", "log_loss"]:
    ranked = (
        overall_ranks[overall_ranks["metric"] == metric]
        .sort_values("rank")
        .drop(columns="metric")
        .reset_index(drop=True)
    )
    ranked["model"] = ranked["model"].map(format_model_label)
    print(f"\n{rank_metric_labels[metric]}")
    display(ranked.style.format({"rank": "{:.2f}"}).background_gradient(subset=["rank"], cmap="Greens_r"))


## Normalized metrics plots

In [ ]:
import seaborn as sns

from pfns.experiments.model_benchmarks.analysis import (
    add_normalized_comparison_metrics,
    add_numeric_buckets,
)

DATASET_SIZE_BUCKET_Q = 5

metric_specs = [
    ("normalized_accuracy", "Normalized Accuracy \u2191"),
    ("normalized_roc_auc", "Normalized ROC-AUC \u2191"),
    ("normalized_log_loss", "Normalized CE \u2191"),
]
raw_metric_cols = ["accuracy", "roc_auc", "log_loss"]

def _format_bucket_bound(value: float) -> str:
    value = int(round(float(value)))
    if value >= 1_000_000:
        short = f"{value / 1_000_000:.1f}M"
        return short.replace(".0M", "M")
    if value >= 1000:
        short = f"{value / 1000:.1f}K"
        return short.replace(".0K", "K")
    return f"{value}"

if "dataset_num_rows" not in all_results.columns:
    raise RuntimeError("all_results does not include dataset_num_rows, so dataset-size analysis cannot be plotted.")

dataset_metric_base = per_dataset.rename(
    columns={
        "dataset_num_rows_first": "dataset_num_rows",
        "accuracy_mean": "accuracy",
        "roc_auc_mean": "roc_auc",
        "log_loss_mean": "log_loss",
    }
)[["model", "dataset", "dataset_num_rows", *raw_metric_cols]].dropna(
    subset=["dataset_num_rows", *raw_metric_cols]
)

normalized_dataset_size_df = add_normalized_comparison_metrics(
    dataset_metric_base.melt(
        id_vars=["model", "dataset", "dataset_num_rows"],
        value_vars=raw_metric_cols,
        var_name="metric",
        value_name="value",
    ).dropna(subset=["dataset_num_rows", "value"]).assign(rep=lambda df: df["dataset"]),
    metric_keys=raw_metric_cols,
    higher_is_better_metrics={"accuracy", "roc_auc"},
    group_cols=("dataset",),
)
normalized_dataset_size_df = add_numeric_buckets(
    normalized_dataset_size_df[
        normalized_dataset_size_df["metric"].isin([metric for metric, _ in metric_specs])
    ].copy(),
    value_col="dataset_num_rows",
    bucket_col="size_bucket",
    q=DATASET_SIZE_BUCKET_Q,
).dropna(subset=["size_bucket"])
bucket_meta = (
    normalized_dataset_size_df.groupby("size_bucket", observed=True)
    .agg(
        min_rows=("dataset_num_rows", "min"),
        max_rows=("dataset_num_rows", "max"),
        n_datasets=("dataset", "nunique"),
    )
    .reset_index()
)
bucket_label_map = {
    row["size_bucket"]: (
        f"{_format_bucket_bound(row['min_rows'])} - {_format_bucket_bound(row['max_rows'])}"
        f"\nn={int(row['n_datasets'])}"
    )
    for _, row in bucket_meta.iterrows()
}
normalized_dataset_size_df["size_bucket"] = normalized_dataset_size_df["size_bucket"].cat.rename_categories(bucket_label_map)
normalized_dataset_size_df["bucket_idx"] = normalized_dataset_size_df["size_bucket"].cat.codes
bucket_order = normalized_dataset_size_df["size_bucket"].cat.categories.tolist()
bucket_summary = (
    normalized_dataset_size_df.groupby(["metric", "model", "size_bucket"], observed=True)["value"]
    .mean()
    .reset_index()
)
model_order = [model_name for model_name in summary.index.tolist() if model_name in set(bucket_summary["model"])]

fig, axes = plt.subplots(
    1,
    len(metric_specs),
    figsize=(24, max(5.0, 0.45 * len(model_order) + 1.5)),
    dpi=300,
    constrained_layout=True,
)
if len(metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, metric_specs):
    heatmap_df = (
        bucket_summary[bucket_summary["metric"] == metric_key]
        .assign(size_bucket=lambda df: df["size_bucket"].astype(str))
        .pivot(index="model", columns="size_bucket", values="value")
        .reindex(index=model_order, columns=bucket_order)
    )
    heatmap_df.index = [format_model_label(model_name) for model_name in heatmap_df.index]
    sns.heatmap(
        heatmap_df,
        ax=ax,
        cmap="viridis",
        vmin=0.0,
        vmax=1.0,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        linecolor="white",
        cbar=ax is axes[-1],
        cbar_kws={"label": "Mean normalized score"},
        annot_kws={"fontsize": 7},
    )
    ax.set_title(metric_title)
    ax.set_xlabel("")
    ax.set_ylabel("Model" if ax is axes[0] else "")
    ax.tick_params(axis="x", labelrotation=0, labelsize=8, pad=6)

plt.show()


In [ ]:
import numpy as np
from pathlib import Path


def _find_repo_root(start_path=Path.cwd()):
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "PFNs" / "notebooks" / "real_world_experiments.ipynb").exists():
            return candidate
    return Path.cwd().resolve()


GRAPHICS_DIR = _find_repo_root() / "PFNs" / "graphics"

from pfns.experiments.model_benchmarks.plotting import (
    make_display_name_formatter,
    resolve_display_name_map,
)

REAL_WORLD_SEQ_LEN_LABEL_OVERRIDES = {
    "Softmax_Transformer": "Softmax Non-Causal",
    "Transformer_Non_Causal": "Softmax Non-Causal",
    "equal_params:Transformer_Comb_ST": "Softmax Non-Causal",
    "Linear_Attention_Non_Causal": "Linear Non-Causal",
    "DeltaNet_Comb_ST": "DeltaNet",
    "DeltaNet_Comb_ST_Reference_New": "DeltaNet",
    "oracles:DeltaNet_Comb_ST": "DeltaNet",
}
REAL_WORLD_SEQ_LEN_STYLE_OVERRIDES = {
    "Softmax_Transformer": ("s", "-", "#08519C"),
    "Transformer_Non_Causal": ("s", "-", "#08519C"),
    "equal_params:Transformer_Comb_ST": ("s", "-", "#08519C"),
    "Linear_Attention_Non_Causal": ("<", "-", "#00796B"),
    "DeltaNet_Comb_ST": ("v", "-", "#B2182B"),
    "DeltaNet_Comb_ST_Reference_New": ("v", "-", "#B2182B"),
    "oracles:DeltaNet_Comb_ST": ("v", "-", "#B2182B"),
}

display_name_map = resolve_display_name_map(all_results)
display_name_map.update(REAL_WORLD_SEQ_LEN_LABEL_OVERRIDES)
format_model_label = make_display_name_formatter(display_name_map=display_name_map)
style_map.update(
    {
        model_name: style
        for model_name, style in REAL_WORLD_SEQ_LEN_STYLE_OVERRIDES.items()
        if model_name in style_map or model_name in set(model_order)
    }
)
model_colors = {name: style_map[name][2] for name in model_order if name in style_map}


def _with_real_world_seq_len_display_names(df):
    plot_df = df.copy()
    plot_df["display_name"] = plot_df["model"].map(format_model_label)
    return plot_df


curve_df = _with_real_world_seq_len_display_names(
    normalized_dataset_size_df[
        normalized_dataset_size_df["model"].isin(model_order)
    ]
)

fig, axes = plot_curves_from_df(
    curve_df,
    specs=metric_specs,
    style_map=style_map,
    x_col="bucket_idx",
    value_col="value",
    x_label="Dataset size bucket",
    title_suffix=" by Dataset Size Bucket",
    figsize=(24, 7),
    show=False,
)
if axes is not None:
    for ax, (_, metric_title) in zip(axes, metric_specs):
        ax.set_ylabel(metric_title)
        ax.set_xticks(range(len(bucket_order)))
        ax.set_xticklabels(bucket_order, rotation=0, ha="center", fontsize=9)
        ax.set_xlabel("Dataset size split")
        ax.set_ylim(-0.05, 1.05)
        ax.grid(axis="y", alpha=0.25)

plt.show()

raw_metric_specs = [
    ("accuracy", "Accuracy ↑"),
    ("roc_auc", "ROC-AUC ↑"),
    ("log_loss", "CE ↓"),
]
bucket_lookup = normalized_dataset_size_df[
    ["dataset", "dataset_num_rows", "size_bucket", "bucket_idx"]
].drop_duplicates()
raw_curve_df = (
    dataset_metric_base[dataset_metric_base["model"].isin(model_order)]
    .melt(
        id_vars=["model", "dataset", "dataset_num_rows"],
        value_vars=raw_metric_cols,
        var_name="metric",
        value_name="value",
    )
    .merge(bucket_lookup, on=["dataset", "dataset_num_rows"], how="inner")
    .dropna(subset=["value", "bucket_idx"])
)
raw_curve_df = _with_real_world_seq_len_display_names(raw_curve_df)
raw_bucket_means = (
    raw_curve_df.groupby(["metric", "model", "bucket_idx"], observed=True)["value"]
    .mean()
    .reset_index()
)

fig, axes = plot_curves_from_df(
    raw_curve_df,
    specs=raw_metric_specs,
    style_map=style_map,
    x_col="bucket_idx",
    value_col="value",
    x_label="Dataset size bucket",
    title_suffix=" by Dataset Size Bucket",
    figsize=(24, 7),
    show=False,
)
if axes is not None:
    for ax, (metric_key, metric_title) in zip(axes, raw_metric_specs):
        ax.set_ylabel(metric_title)
        ax.set_xticks(range(len(bucket_order)))
        ax.set_xticklabels(bucket_order, rotation=0, ha="center", fontsize=9)
        ax.set_xlabel("Dataset size split")
        metric_values = raw_bucket_means.loc[raw_bucket_means["metric"] == metric_key, "value"].dropna()
        if not metric_values.empty:
            y_min = float(metric_values.min())
            y_max = float(metric_values.max())
            y_span = y_max - y_min
            pad = max(0.15 * y_span, 0.005 if metric_key in {"accuracy", "roc_auc"} else 0.015)
            lower = y_min - pad
            upper = y_max + pad
            if metric_key in {"accuracy", "roc_auc"}:
                lower = max(0.0, lower)
                upper = min(1.0, upper)
            else:
                lower = max(0.0, lower)
            ax.set_ylim(lower, upper)
        ax.grid(axis="y", alpha=0.25)

plt.show()

per_dataset_df = raw_curve_df[raw_curve_df["dataset_num_rows"].gt(0)].copy()

fig, axes = plt.subplots(
    1,
    len(raw_metric_specs),
    figsize=(24, 7),
    dpi=400,
    sharey=False,
)
if len(raw_metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, raw_metric_specs):
    metric_df = per_dataset_df[per_dataset_df["metric"] == metric_key]
    model_count = max(1, len(model_order))
    for model_name in model_order:
        model_df = metric_df[metric_df["model"] == model_name]
        if model_df.empty:
            continue
        marker, _, color = style_map.get(model_name, ("o", "-", None))
        model_idx = model_order.index(model_name)
        dodge = 1.0 + (model_idx - (model_count - 1) / 2.0) * 0.045
        ax.scatter(
            model_df["dataset_num_rows"] * dodge,
            model_df["value"],
            marker=marker,
            facecolors=color,
            edgecolors="#1f1f1f",
            s=56,
            alpha=0.82,
            linewidths=0.45,
            label=format_model_label(model_name),
            zorder=2 + model_idx,
        )
    ax.set_xscale("log")
    ax.set_title(f"{metric_title} per Dataset")
    ax.set_xlabel("Dataset size (rows, log scale)")
    if False and metric_key in {"accuracy", "roc_auc"}:
        ax.set_ylim(-0.03, 1.03)
    else:
        metric_values = metric_df["value"].dropna()
        if not metric_values.empty:
            y_min = float(metric_values.min())
            y_max = float(metric_values.max())
            y_span = y_max - y_min
            pad = max(0.08 * y_span, 0.05)
            ax.set_ylim(max(0.0, y_min - pad), y_max + pad)
    ax.grid(True, which="both", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Performance")

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    legend_cols = min(max(1, len(handles)), 4)
    fig.subplots_adjust(bottom=0.24, top=0.86, wspace=0.18)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=legend_cols,
        fontsize=11,
        frameon=True,
    )
fig.suptitle("Per-Dataset Performance by Dataset Size", y=0.98, fontsize=16)

plt.show()


normalized_per_dataset_df = curve_df[curve_df["dataset_num_rows"].gt(0)].copy()

fig, axes = plt.subplots(
    1,
    len(metric_specs),
    figsize=(24, 7),
    dpi=400,
    sharey=True,
)
if len(metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, metric_specs):
    metric_df = normalized_per_dataset_df[normalized_per_dataset_df["metric"] == metric_key]
    model_count = max(1, len(model_order))
    for model_name in model_order:
        model_df = metric_df[metric_df["model"] == model_name]
        if model_df.empty:
            continue
        marker, _, color = style_map.get(model_name, ("o", "-", None))
        model_idx = model_order.index(model_name)
        dodge = 1.0 # + (model_idx - (model_count - 1) / 2.0) * 0.045
        ax.scatter(
            model_df["dataset_num_rows"] * dodge,
            model_df["value"],
            marker=marker,
            facecolors=color,
            edgecolors="#1f1f1f",
            s=56,
            alpha=0.82,
            linewidths=0.45,
            label=format_model_label(model_name),
            zorder=2 + model_idx,
        )
    ax.set_xscale("log")
    ax.set_title(f"{metric_title} per Dataset")
    ax.set_xlabel("Dataset size (rows, log scale)")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, which="both", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Normalized performance")

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    legend_cols = min(max(1, len(handles)), 4)
    fig.subplots_adjust(bottom=0.24, top=0.86, wspace=0.18)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=legend_cols,
        fontsize=11,
        frameon=True,
    )
fig.suptitle("Per-Dataset Normalized Performance by Dataset Size", y=0.98, fontsize=16)

plt.show()

REFERENCE_MODEL_FOR_GAIN_PLOT = "Softmax_Transformer"
gain_models = [model_name for model_name in model_order if model_name != REFERENCE_MODEL_FOR_GAIN_PLOT]
reference_df = raw_curve_df[
    raw_curve_df["model"] == REFERENCE_MODEL_FOR_GAIN_PLOT
][["dataset", "metric", "value"]].rename(columns={"value": "reference_value"})
if reference_df.empty:
    raise RuntimeError(f"Reference model {REFERENCE_MODEL_FOR_GAIN_PLOT!r} is not available for the ratio plot.")

gain_df = raw_curve_df[raw_curve_df["model"].isin(gain_models)].merge(
    reference_df,
    on=["dataset", "metric"],
    how="inner",
).dropna(subset=["value", "reference_value", "dataset_num_rows"])
higher_is_better_mask = gain_df["metric"].isin(["accuracy", "roc_auc"])
gain_df["gain"] = gain_df["reference_value"] - gain_df["value"]
gain_df.loc[higher_is_better_mask, "gain"] = (
    gain_df.loc[higher_is_better_mask, "value"]
    - gain_df.loc[higher_is_better_mask, "reference_value"]
)

reference_model_label = format_model_label(REFERENCE_MODEL_FOR_GAIN_PLOT)
ratio_metric_specs = [
    ("accuracy", f"Acc / {reference_model_label} Acc ↑"),
    ("roc_auc", f"ROC-AUC / {reference_model_label} ROC-AUC ↑"),
    ("log_loss", f"{reference_model_label} CE / CE ↑"),
]
ratio_df = gain_df[gain_df["reference_value"].gt(0) & gain_df["value"].gt(0)].copy()
ratio_df["ratio"] = ratio_df["value"] / ratio_df["reference_value"]
lower_is_better_mask = ratio_df["metric"] == "log_loss"
ratio_df.loc[lower_is_better_mask, "ratio"] = (
    ratio_df.loc[lower_is_better_mask, "reference_value"]
    / ratio_df.loc[lower_is_better_mask, "value"]
)

fig, axes = plt.subplots(
    1,
    len(ratio_metric_specs),
    figsize=(24, 7),
    dpi=400,
    sharey=False,
)
if len(ratio_metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, ratio_metric_specs):
    metric_df = ratio_df[ratio_df["metric"] == metric_key]
    model_count = max(1, len(gain_models))
    for model_name in gain_models:
        model_df = metric_df[metric_df["model"] == model_name]
        if model_df.empty:
            continue
        marker, _, color = style_map.get(model_name, ("o", "-", None))
        model_idx = gain_models.index(model_name)
        dodge = 1.0 # + (model_idx - (model_count - 1) / 2.0) * 0.045
        ax.scatter(
            model_df["dataset_num_rows"] * dodge,
            model_df["ratio"],
            marker=marker,
            facecolors=color,
            edgecolors="#1f1f1f",
            s=56,
            alpha=0.82,
            linewidths=0.45,
            label=format_model_label(model_name),
            zorder=2 + model_idx,
        )
    ratio_values = metric_df["ratio"].dropna()
    if not ratio_values.empty:
        y_min = min(float(ratio_values.min()), 1.0)
        y_max = max(float(ratio_values.max()), 1.0)
        y_span = y_max - y_min
        pad = max(0.10 * y_span, 0.02)
        ax.set_ylim(max(0.0, y_min - pad), y_max + pad)
    ax.axhline(1.0, color="0.25", linestyle="--", linewidth=1.0, alpha=0.8)
    ax.set_xscale("log")
    ax.set_title(f"{metric_title} per Dataset")
    ax.set_xlabel("Dataset size (rows, log scale)")
    ax.grid(True, which="both", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel(f"Performance ratio vs {reference_model_label}")

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    legend_cols = min(max(1, len(handles)), 4)
    fig.subplots_adjust(bottom=0.24, top=0.86, wspace=0.18)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=legend_cols,
        fontsize=11,
        frameon=True,
    )
fig.suptitle(f"Per-Dataset Performance Ratio vs {reference_model_label} by Dataset Size", y=0.98, fontsize=16)

plt.show()

ranking_metric_specs = [
    ("roc_auc", "ROC-AUC Rank"),
    ("accuracy", "Accuracy Rank"),
]
dataset_order_df = (
    raw_curve_df[["dataset", "dataset_num_rows"]]
    .drop_duplicates()
    .sort_values(["dataset_num_rows", "dataset"])
    .reset_index(drop=True)
)
dataset_order = dataset_order_df["dataset"].tolist()
dataset_to_y = {dataset: idx for idx, dataset in enumerate(dataset_order)}
dataset_label_map = {
    row["dataset"]: (
        f"{str(row['dataset'])[:34]}{'...' if len(str(row['dataset'])) > 34 else ''}"
        f" ({_format_bucket_bound(row['dataset_num_rows'])})"
    )
    for _, row in dataset_order_df.iterrows()
}

rank_plot_frames = []
for metric_key, _ in ranking_metric_specs:
    metric_df = raw_curve_df[raw_curve_df["metric"] == metric_key].copy()
    metric_df["rank"] = metric_df.groupby("dataset", observed=True)["value"].rank(
        method="average",
        ascending=False,
    )
    metric_df["dataset_y"] = metric_df["dataset"].map(dataset_to_y)
    rank_plot_frames.append(metric_df)
rank_plot_df = pd.concat(rank_plot_frames, ignore_index=True)

fig, axes = plt.subplots(
    1,
    len(ranking_metric_specs),
    figsize=(17, max(7.0, 0.28 * len(dataset_order) + 2.0)),
    dpi=400,
    sharey=True,
)
if len(ranking_metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, ranking_metric_specs):
    metric_df = rank_plot_df[rank_plot_df["metric"] == metric_key]
    for model_name in model_order:
        model_df = metric_df[metric_df["model"] == model_name].sort_values("dataset_y")
        if model_df.empty:
            continue
        marker, _, color = style_map.get(model_name, ("o", "-", None))
        model_idx = model_order.index(model_name)
        ax.plot(
            model_df["rank"],
            model_df["dataset_y"],
            color=color,
            linewidth=1.3,
            alpha=0.45,
            zorder=1,
        )
        ax.scatter(
            model_df["rank"],
            model_df["dataset_y"],
            marker=marker,
            facecolors=color,
            edgecolors="#1f1f1f",
            s=54,
            alpha=0.86,
            linewidths=0.45,
            label=format_model_label(model_name),
            zorder=2 + model_idx,
        )
    ax.set_title(f"{metric_title} by Dataset")
    ax.set_xlabel("Rank (1 = best)")
    ax.set_xlim(0.5, len(model_order) + 0.5)
    ax.set_xticks(range(1, len(model_order) + 1))
    ax.grid(True, axis="x", alpha=0.25)
    ax.grid(True, axis="y", alpha=0.12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_yticks(range(len(dataset_order)))
axes[0].set_yticklabels([dataset_label_map[dataset] for dataset in dataset_order], fontsize=7)
axes[0].set_ylabel("Dataset (rows)")
axes[0].invert_yaxis()

fig.subplots_adjust(bottom=0.125, top=0.91, left=0.24, right=0.99, wspace=0.08)
handles, labels = axes[0].get_legend_handles_labels()
if handles:
    legend_cols = min(max(1, len(handles)), 4)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=legend_cols,
        fontsize=14,
        frameon=True,
        markerscale=1.6,
        handlelength=1.4,
        handletextpad=0.8,
        columnspacing=1.8,
        borderpad=0.55,
    )
fig.suptitle("Per-Dataset Model Ranking", y=0.98, fontsize=16)

ranking_pdf_path = GRAPHICS_DIR / "real_world_per_dataset_model_ranking.pdf"
ranking_pdf_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(ranking_pdf_path, bbox_inches="tight", pad_inches=0.02)
print(f"Saved ranking figure to {ranking_pdf_path}")

plt.show()

fig, axes = plt.subplots(
    1,
    len(ranking_metric_specs),
    figsize=(24, 7),
    dpi=400,
    sharey=True,
)
if len(ranking_metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, ranking_metric_specs):
    metric_df = rank_plot_df[rank_plot_df["metric"] == metric_key]
    for model_name in model_order:
        model_df = metric_df[metric_df["model"] == model_name].sort_values("dataset_num_rows")
        if model_df.empty:
            continue
        marker, linestyle, color = style_map.get(model_name, ("o", "-", None))
        ax.plot(
            model_df["dataset_num_rows"],
            model_df["rank"],
            marker=marker,
            linestyle=linestyle,
            color=color,
            linewidth=2.2,
            markersize=6.0,
            alpha=0.9,
            label=format_model_label(model_name),
        )
    ax.set_title(f"{metric_title} Across Datasets")
    ax.set_xlabel("Dataset size (rows, log scale)")
    ax.set_xscale("log")
    ax.set_ylim(len(model_order) + 0.25, 0.75)
    ax.set_yticks(range(1, len(model_order) + 1))
    ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Rank (1 = best)")

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    legend_cols = min(max(1, len(handles)), 4)
    fig.subplots_adjust(bottom=0.22, top=0.86, wspace=0.18)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=legend_cols,
        fontsize=11,
        frameon=True,
    )
fig.suptitle("Connected Per-Dataset Model Rankings", y=0.98, fontsize=16)

plt.show()

NORMALIZED_SCORE_ROLLING_WINDOW = 5
rolling_metric_specs = [
    ("normalized_roc_auc", "Normalized ROC-AUC ↑"),
    ("normalized_accuracy", "Normalized Accuracy ↑"),
]
rolling_metric_keys = {metric_key for metric_key, _ in rolling_metric_specs}
rolling_score_frames = []
for (metric_key, model_name), model_df in curve_df[
    curve_df["metric"].isin(rolling_metric_keys)
].groupby(["metric", "model"], observed=True):
    model_df = model_df[model_df["dataset_num_rows"].gt(0)].sort_values("dataset_num_rows").copy()
    if model_df.empty:
        continue
    rolling_score = model_df["value"].rolling(
        window=NORMALIZED_SCORE_ROLLING_WINDOW,
        center=True,
        min_periods=1,
    )
    model_df["score_roll_mean"] = rolling_score.mean()
    model_df["score_roll_std"] = rolling_score.std().fillna(0.0)
    rolling_score_frames.append(model_df)
rolling_score_df = pd.concat(rolling_score_frames, ignore_index=True)

fig, axes = plt.subplots(
    1,
    len(rolling_metric_specs),
    figsize=(16, 6.2),
    dpi=400,
    sharey=True,
)
if len(rolling_metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, rolling_metric_specs):
    metric_df = rolling_score_df[rolling_score_df["metric"] == metric_key]
    for model_name in model_order:
        model_df = metric_df[metric_df["model"] == model_name].sort_values("dataset_num_rows")
        if model_df.empty:
            continue
        marker, linestyle, color = style_map.get(model_name, ("o", "-", None))
        x = model_df["dataset_num_rows"].to_numpy(dtype=float)
        y = model_df["score_roll_mean"].to_numpy(dtype=float)
        y_std = model_df["score_roll_std"].to_numpy(dtype=float)
        # ax.scatter(
        #     x,
        #     model_df["value"],
        #     marker=marker,
        #     color=color,
        #     s=22,
        #     alpha=0.18,
        #     linewidths=0,
        #     zorder=1,
        # )
        ax.fill_between(
            x,
            np.clip(y - y_std, 0.0, 1.0),
            np.clip(y + y_std, 0.0, 1.0),
            color=color,
            alpha=0.12,
            linewidth=0,
            zorder=2,
        )
        ax.plot(
            x,
            y,
            marker=marker,
            linestyle=linestyle,
            color=color,
            linewidth=2.6,
            markersize=6.0,
            label=format_model_label(model_name),
            zorder=3,
        )
    ax.set_title(f"{metric_title} Rolling Mean")
    ax.set_xlabel("Dataset size (rows, log scale)")
    ax.set_xscale("log")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Rolling normalized score")

fig.subplots_adjust(bottom=0.22, top=0.86, left=0.08, right=0.99, wspace=0.10)
handles, labels = axes[0].get_legend_handles_labels()
if handles:
    legend_cols = min(max(1, len(handles)), 4)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=legend_cols,
        fontsize=11,
        frameon=True,
    )
fig.suptitle(
    f"Rolling Normalized Scores (window={NORMALIZED_SCORE_ROLLING_WINDOW} datasets)",
    y=0.98,
    fontsize=16,
)

rolling_pdf_path = GRAPHICS_DIR / "real_world_rolling_normalized_scores.pdf"
rolling_pdf_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(rolling_pdf_path, bbox_inches="tight", pad_inches=0.02)
print(f"Saved rolling normalized score figure to {rolling_pdf_path}")

plt.show()


## Setting Analysis Shared Setup

Shared helpers/constants used by both cross-mode ranking and gain/significance sections.


In [ ]:
from pfns.experiments.model_benchmarks.setting_analysis import (
    SETTING_METRIC_DIRECTION,
    SETTING_METRIC_LABELS,
    get_setting_preprocess,
)
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis
from pfns.experiments.model_benchmarks.plotting import (
    make_display_name_formatter,
    resolve_display_name_map,
)

display_name_map = resolve_display_name_map(all_results)
format_model_label = make_display_name_formatter(display_name_map=display_name_map)


## Cross-Mode Setting Ranking

Rank and average `Comb_MT`, `Comb_ST`, `Int_ST`, `Int_MT` across model types using only canonical model names.

Models with extra suffixes (for example `short_conv`, hidden-size sweeps, `Layers_24`, or seq-len variants) are ignored automatically.


In [ ]:
import pandas as pd

if per_dataset is None or per_dataset.empty:
    raise RuntimeError("No per-dataset results are available for setting ranking.")

TARGET_SETTINGS = ["Comb_MT", "Comb_ST", "Int_ST", "Int_MT"]

pre = get_setting_preprocess(
    results_df=per_dataset,
    target_settings=TARGET_SETTINGS,
)

evaluated_df = pre["model_meta"]
evaluated_presence = pre["presence"].reindex(columns=TARGET_SETTINGS, fill_value=False).sort_index()
evaluated_complete_types = pre["eligible_model_types"]
analysis_df = pre["filtered_results"].copy()

print("Canonical setting availability in currently loaded evaluation results.")
display(evaluated_presence)
print(f"Model types with all four settings evaluated: {evaluated_complete_types}")

metric_specs = [
    ("accuracy_mean", True, "Accuracy"),
    ("roc_auc_mean", True, "ROC-AUC"),
    ("log_loss_mean", False, "CE"),
    ("ece_mean", False, "ECE"),
]

missing_metric_cols = [col for col, _, _ in metric_specs if col not in analysis_df.columns]
if missing_metric_cols:
    raise RuntimeError(f"Missing required metric columns for ranking: {missing_metric_cols}")

rank_frames = []
for metric_col, higher_is_better, metric_label in metric_specs:
    metric_rank_df = analysis_df[["dataset", "model_type", "setting", metric_col]].dropna().copy()
    metric_rank_df["rank"] = metric_rank_df.groupby(
        ["model_type", "dataset"], observed=True
    )[metric_col].rank(method="average", ascending=not higher_is_better)
    metric_rank_df["metric"] = metric_label
    rank_frames.append(metric_rank_df[["metric", "setting", "rank"]])

rank_df = pd.concat(rank_frames, ignore_index=True)
rank_summary = (
    rank_df.groupby(["metric", "setting"], observed=True)
    .agg(mean_rank=("rank", "mean"), std_rank=("rank", "std"), n_pairs=("rank", "size"))
    .reset_index()
)

rank_table = (
    rank_summary
    .pivot(index="setting", columns="metric", values="mean_rank")
    .reindex(TARGET_SETTINGS)
)

print("Mean rank by setting across model types and datasets (lower is better).")
display(rank_table.style.format("{:.3f}").background_gradient(cmap="Greens_r", axis=0))

overall_setting_rank = (
    rank_summary.groupby("setting", observed=True)["mean_rank"]
    .mean()
    .reindex(TARGET_SETTINGS)
    .sort_values()
)
print("Overall mean rank across all metrics (lower is better).")
display(overall_setting_rank.to_frame("overall_mean_rank").style.format({"overall_mean_rank": "{:.3f}"}).background_gradient(cmap="Greens_r"))

setting_metric_means = (
    analysis_df.groupby("setting", observed=True)
    .agg(
        accuracy_mean=("accuracy_mean", "mean"),
        roc_auc_mean=("roc_auc_mean", "mean"),
        log_loss_mean=("log_loss_mean", "mean"),
        ece_mean=("ece_mean", "mean"),
    )
    .reindex(TARGET_SETTINGS)
)
print("Raw metric averages by setting across eligible model types.")
display(setting_metric_means.style.format({
    "accuracy_mean": "{:.4f}",
    "roc_auc_mean": "{:.4f}",
    "log_loss_mean": "{:.4f}",
    "ece_mean": "{:.4f}",
}))

## Gain And Significance Across Settings

Paired gain and interval analysis for `Comb_MT`, `Comb_ST`, `Int_MT`, `Int_ST` (not model-vs-model).

Positive gain always means the compared setting is better than the reference setting.


In [ ]:
import matplotlib.pyplot as plt

SETTING_ANALYSIS = {
    "metric": "roc_auc",  # one of: accuracy, roc_auc, log_loss, ece
    "unit": "dataset",  # dataset or split
    "target_labels": ["Comb_MT", "Comb_ST", "Int_MT", "Int_ST"],
    "reference_label": "Int_MT",
    "error_bars": "ci95",  # std or ci95
    "compare_col": "setting",
    "comparison_label": "setting",
    "wilcoxon_alpha": 0.05,
}

if all_results is None or all_results.empty:
    raise RuntimeError("No aggregated all_results dataframe is available for setting CD analysis.")

metric = SETTING_ANALYSIS["metric"]
if metric not in SETTING_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SETTING_METRIC_DIRECTION)}")
if metric not in all_results.columns:
    raise RuntimeError(f"Metric column {metric!r} is not present in all_results.")

SETTING_DISPLAY_NAMES = {
    "Comb_MT": "Combined \n Multi Target",
    "Comb_ST": "Combined \n Single Target",
    "Int_MT": "Interleaved Multi Target",
    "Int_ST": "Interleaved \n Single Target",
}
SETTING_DISPLAY_NAMES_MULTILINE = {
    "Combined Multi Target": "Combined\nMulti Target",
    "Combined Single Target": "Combined\nSingle Target",
    "Interleaved Multi Target": "Interleaved\nMulti Target",
    "Interleaved Single Target": "Interleaved\nSingle Target",
}

unit = SETTING_ANALYSIS["unit"]
target_labels = list(dict.fromkeys(SETTING_ANALYSIS["target_labels"]))
target_display_labels = [SETTING_DISPLAY_NAMES.get(label, label) for label in target_labels]
reference_display_label = SETTING_DISPLAY_NAMES.get(SETTING_ANALYSIS["reference_label"], SETTING_ANALYSIS["reference_label"])

pre = get_setting_preprocess(
    results_df=all_results,
    target_settings=target_labels,
)
presence = pre["presence"].reindex(columns=target_labels, fill_value=False).rename(columns=SETTING_DISPLAY_NAMES)
eligible_model_types = pre["eligible_model_types"]
comparison_results = pre["filtered_results"]
comparison_results = comparison_results.copy()
comparison_results["setting"] = comparison_results["setting"].map(lambda label: SETTING_DISPLAY_NAMES.get(label, label))

print("Eligible model types (all requested settings available):", eligible_model_types)
print("Setting availability by model type:")
display(presence)

higher_is_better = SETTING_METRIC_DIRECTION[metric]
pair_cols = ["model_type", "dataset"] if unit == "dataset" else ["model_type", "dataset", "split"]

analysis = run_comparison_analysis(
    comparison_results=comparison_results,
    metric_col=metric,
    metric_label=SETTING_METRIC_LABELS[metric],
    compare_col=SETTING_ANALYSIS["compare_col"],
    target_labels=target_display_labels,
    pair_cols=pair_cols,
    higher_better=higher_is_better,
    reference_label=reference_display_label,
    unit=unit,
    error_bars=SETTING_ANALYSIS["error_bars"],
    comparison_label=SETTING_ANALYSIS["comparison_label"],
    include_boxplot=True,
    include_pairwise_tables=True,
    include_cd_diagram=True,
    wilcoxon_alpha=SETTING_ANALYSIS["wilcoxon_alpha"],
    empty_error_message=(
        "No complete paired rows found across all requested settings. "
        "Try unit='dataset' or evaluate more models/settings."
    ),
)

print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
print(f"Metric: {SETTING_METRIC_LABELS[metric]} ({metric})")
print(f"Pairing unit: {unit}")
print(f"Reference setting: {analysis['gain']['reference_label']}")
print(f"Comparisons: {analysis['gain']['comparison_labels']}")
print("Setting means:")
display(analysis["gain"]["label_means"].to_frame("mean"))

print("\nPaired gain summary by setting (positive means better than reference setting):")
display(
    analysis["gain"]["gain_summary"].style.format(
        {
            "mean_gain": "{:+.5f}",
            "std_gain": "{:.5f}",
            "sem_gain": "{:.5f}",
            "ci95": "{:.5f}",
            "ci95_low": "{:+.5f}",
            "ci95_high": "{:+.5f}",
            "n_pairs": "{:.0f}",
            "share_pairs_better": "{:.1%}",
        }
    ).background_gradient(subset=["mean_gain"], cmap="RdYlGn")
)

if analysis["pairwise"] is not None:
    print("\nPairwise mean gain matrix (row setting vs column setting, positive means row is better):")
    display(analysis["pairwise"]["pairwise_mean"].style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None))

    print("Pairwise 95% CI half-width matrix:")
    display(analysis["pairwise"]["pairwise_ci95"].style.format("{:.4f}").background_gradient(cmap="Blues", axis=None))

    print("Pairwise significance proxy (95% CI excludes zero):")
    display(analysis["pairwise"]["pairwise_sig"])

for fig_key in ["gain_barh", "gain_boxplot", "wilcoxon_cd"]:
    if fig_key in analysis["figures"]:
        fig, _ = analysis["figures"][fig_key]
        if fig_key == "gain_barh" and fig.axes:
            fig.axes[0].set_title(f"Paired gain vs {reference_display_label} with 95% confidence intervals")
        if fig_key == "wilcoxon_cd" and fig.axes:
            for text in fig.axes[0].texts:
                if text.get_position()[0] > 0.5:
                    label = text.get_text()
                    text.set_text(SETTING_DISPLAY_NAMES_MULTILINE.get(label, label))
        display(fig)
        plt.close(fig)




## Training setup LaTeX table

Generate a compact LaTeX table for the aggregated training-setup comparison.

In [ ]:
from pfns.experiments.model_benchmarks.latex_tables import (
    build_setting_metric_tables,
    render_setting_average_performance_latex,
)

SETTING_LATEX_METRICS = [
    ("roc_auc", r"ROC-AUC $\uparrow$", True),
    ("accuracy", r"ACC $\uparrow$", True),
    ("log_loss", r"CE $\downarrow$", False),
]
SETTING_LATEX_BENCHMARKS = {
    "openml": r"\shortstack{OpenML CC-18\\(filtered \& subsampled)}",
    "tabarena": r"\shortstack{TabArena\\(feature subsampled ($\le 20$))}",
}
SETTING_LATEX_PAIR_COLS = ["model_type", "dataset"]
SETTING_LATEX_ADD_RAW_SIGNIFICANCE_MARKERS = True
SETTING_LATEX_ADD_SIGNIFICANCE_MARKERS = True
SETTING_LATEX_SIGNIFICANCE_ALPHA = 0.05
SETTING_LATEX_BOLD_RANKS = (1,)
SETTING_LATEX_UNDERLINE_RANKS = ()
SETTING_LATEX_RANK_COLORS = {}
SETTING_LATEX_SEQUENCE_LENGTH_LABELS = {
    "Combined Multi Target": "1x",
    "Combined Single Target": "1x",
    "Interleaved Multi Target": "2x",
    "Interleaved Single Target": "2x",
}
SETTING_LATEX_SEQUENCE_LENGTH_COLUMN = "Seq. len."

SETTING_LATEX_DATA_DESCRIPTION = (
    "on filtered and subsampled datasets from the OpenML CC-18 and TabArena benchmarks. "
    "For OpenML CC-18, we use the 30-dataset subset from TabPFN v1 "
    "\\cite{hollmann2023tabpfn}, with additional subsampling to at most 1000 "
    "samples and 20 features. For TabArena, we use its 38 classification "
    "datasets, with features subsampled to at most 20. For each eligible "
    "model-family and dataset pair, split-level results are first averaged per "
    "setup, and only complete four-setup pairs are retained. "
)
SETTING_LATEX_SIGNIFICANCE_DESCRIPTION = (
    " Superscript letters denote pairwise significance groups within each benchmark metric; "
    "setups sharing at least one letter are not significantly different. Significance "
    "is computed over dataset-level {score_kind} scores after averaging over eligible "
    "model families, using paired two-sided Wilcoxon signed-rank tests with Holm's "
    "alpha (5\\%) correction."
)
def prepare_setting_comparison_results(results_df):
    pre = get_setting_preprocess(
        results_df=results_df,
        target_settings=target_labels,
    )
    prepared = pre["filtered_results"].copy()
    prepared["setting"] = prepared["setting"].map(
        lambda label: SETTING_DISPLAY_NAMES.get(label, label)
    )
    return prepared


def setting_metric_latex_label(metric_col, normalize=False):
    for configured_metric_col, metric_label, _ in SETTING_LATEX_METRICS:
        if configured_metric_col == metric_col:
            if not normalize:
                return metric_label
            metric_name = metric_label.split(" $", maxsplit=1)[0]
            return rf"\shortstack{{{metric_name}\\score}}"
    return SETTING_METRIC_LABELS.get(metric_col, metric_col)


def load_setting_benchmark_results():
    benchmark_results = {}
    missing_by_benchmark = {}
    for preset_name, benchmark_label in SETTING_LATEX_BENCHMARKS.items():
        preset_results, missing_models = load_cached_real_world_results_for_preset(
            preset_name,
            model_names=models_to_compare,
            device=device,
            output_root=OUTPUT_ROOT,
        )
        missing_by_benchmark[benchmark_label] = missing_models
        if preset_results.empty:
            print(f"No compatible cached bundles found for {benchmark_label}.")
            continue
        benchmark_results[preset_name] = preset_results
    return benchmark_results, missing_by_benchmark


def setting_caption(*, normalized, include_significance):
    significance = (
        SETTING_LATEX_SIGNIFICANCE_DESCRIPTION.format(
            score_kind="normalized" if normalized else "raw"
        )
        if include_significance
        else ""
    )
    if not normalized:
        return (
            "\\caption{Aggregated training setup performance "
            + SETTING_LATEX_DATA_DESCRIPTION
            + "Values are averaged across datasets and model families that support all four setups. "
            + "Seq. len. reports the effective sequence length relative to combined modes. "
            + "Bold marks the best setup for each benchmark metric."
            + significance
            + "}"
        )

    return (
        "\\caption{Normalized training setup performance "
        + SETTING_LATEX_DATA_DESCRIPTION
        + "Within each retained pair, each metric is min-max normalized across the four training setups to $[0,1]$. "
        + "For CE, lower raw loss maps to a higher normalized score. Reported values are "
        + "averaged across datasets and model families that support all four setups. Seq. len. "
        + "reports the effective sequence length relative to combined modes. Bold "
        + "marks the best normalized setup score for each benchmark metric."
        + significance
        + "}"
    )


def build_and_render_setting_latex(*, normalize, caption, label, add_significance_markers):
    tables = build_setting_metric_tables(
        benchmark_results=setting_benchmark_results,
        benchmark_labels=SETTING_LATEX_BENCHMARKS,
        metrics=SETTING_LATEX_METRICS,
        target_labels=target_display_labels,
        compare_col=SETTING_ANALYSIS["compare_col"],
        pair_cols=SETTING_LATEX_PAIR_COLS,
        prepare_comparison_results=prepare_setting_comparison_results,
        metric_label_fn=setting_metric_latex_label,
        sort_metric=SETTING_ANALYSIS["metric"],
        normalize=normalize,
        add_significance_markers=add_significance_markers,
        significance_alpha=SETTING_LATEX_SIGNIFICANCE_ALPHA,
    )
    latex = render_setting_average_performance_latex(
        tables[0],
        tables[1],
        tables[2],
        caption=caption,
        label=label,
        rank_colors=SETTING_LATEX_RANK_COLORS,
        bold_ranks=SETTING_LATEX_BOLD_RANKS,
        underline_ranks=SETTING_LATEX_UNDERLINE_RANKS,
        sequence_length_labels=SETTING_LATEX_SEQUENCE_LENGTH_LABELS,
        sequence_length_column_label=SETTING_LATEX_SEQUENCE_LENGTH_COLUMN,
    )
    return (*tables, latex)


setting_benchmark_results, setting_missing_by_benchmark = load_setting_benchmark_results()
(
    setting_performance_table,
    setting_performance_rank_table,
    setting_performance_significance_table,
    setting_n_pairs_by_benchmark,
    setting_performance_latex,
) = build_and_render_setting_latex(
    normalize=False,
    caption=setting_caption(
        normalized=False,
        include_significance=SETTING_LATEX_ADD_RAW_SIGNIFICANCE_MARKERS,
    ),
    label=r"\label{tab:training_setup_performance}",
    add_significance_markers=SETTING_LATEX_ADD_RAW_SIGNIFICANCE_MARKERS,
)
(
    setting_normalized_performance_table,
    setting_normalized_performance_rank_table,
    setting_normalized_performance_significance_table,
    setting_normalized_n_pairs_by_benchmark,
    setting_normalized_performance_latex,
) = build_and_render_setting_latex(
    normalize=True,
    caption=setting_caption(
        normalized=True,
        include_significance=SETTING_LATEX_ADD_SIGNIFICANCE_MARKERS,
    ),
    label=r"\label{tab:training_setup_normalized_performance}",
    add_significance_markers=SETTING_LATEX_ADD_SIGNIFICANCE_MARKERS,
)
print("Average performance LaTeX table:")
print(setting_performance_latex)
print("\nNormalized average performance LaTeX table:")
print(setting_normalized_performance_latex)


## Equal-Params Model Wilcoxon-Holm Comparison

Compare only the registry-defined equal-parameter models using paired Wilcoxon tests with Holm correction.


In [ ]:
import matplotlib.pyplot as plt

from pfns.experiments.model_benchmarks.model_registry import get_models_from_families
from pfns.experiments.model_benchmarks.setting_analysis import (
    SETTING_METRIC_DIRECTION,
    SETTING_METRIC_LABELS,
)
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis

EQUAL_PARAMS_ANALYSIS = {
    "metric": "roc_auc",  # one of: accuracy, roc_auc, log_loss, ece
    "unit": "dataset",  # dataset or split
    "reference_label": None,  # auto-selects best mean metric if None
    "wilcoxon_alpha": 0.05,
}

if all_results is None or all_results.empty:
    raise RuntimeError("No aggregated all_results dataframe is available for equal-params analysis.")

metric = EQUAL_PARAMS_ANALYSIS["metric"]
if metric not in SETTING_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SETTING_METRIC_DIRECTION)}")
if metric not in all_results.columns:
    raise RuntimeError(f"Metric column {metric!r} is not present in all_results.")

unit = EQUAL_PARAMS_ANALYSIS["unit"]
if unit not in {"dataset", "split"}:
    raise ValueError("EQUAL_PARAMS_ANALYSIS['unit'] must be 'dataset' or 'split'.")

registered_equal_models = list(get_models_from_families(["equal_params"]).keys())
available_models = set(all_results["model"].astype(str).unique())
target_models = [model for model in registered_equal_models if model in available_models]

if len(target_models) < 2:
    raise RuntimeError(
        "Need at least two equal-params models in all_results for Wilcoxon-Holm analysis. "
        f"Found {len(target_models)}: {target_models}."
    )

target_display_models = [format_model_label(model) for model in target_models]
comparison_results = all_results[all_results["model"].isin(target_models)].copy()
comparison_results["model"] = comparison_results["model"].map(format_model_label)
if comparison_results.empty:
    raise RuntimeError("No rows found in all_results for equal-params models.")

higher_is_better = SETTING_METRIC_DIRECTION[metric]
model_means = comparison_results.groupby("model", observed=True)[metric].mean().reindex(target_display_models)

reference_label = EQUAL_PARAMS_ANALYSIS["reference_label"]
if reference_label is None:
    reference_label = model_means.idxmax() if higher_is_better else model_means.idxmin()
elif reference_label not in target_models:
    raise RuntimeError(
        f"Configured reference_label={reference_label!r} is not among available equal-params models: {target_models}"
    )
else:
    reference_label = format_model_label(reference_label)

pair_cols = ["dataset"] if unit == "dataset" else ["dataset", "split"]
analysis = run_comparison_analysis(
    comparison_results=comparison_results,
    metric_col=metric,
    metric_label=SETTING_METRIC_LABELS[metric],
    compare_col="model",
    target_labels=target_display_models,
    pair_cols=pair_cols,
    higher_better=higher_is_better,
    reference_label=reference_label,
    unit=unit,
    error_bars="ci95",
    comparison_label="model",
    include_boxplot=False,
    include_pairwise_tables=False,
    include_cd_diagram=True,
    wilcoxon_alpha=EQUAL_PARAMS_ANALYSIS["wilcoxon_alpha"],
    empty_error_message=(
        "No complete paired rows found across equal-params models. "
        "Try unit='dataset' or evaluate more equal-params models."
    ),
)

wilcoxon_summary = analysis["wilcoxon"]
n_pairs = wilcoxon_summary["n_pairs"]

print(f"Metric: {SETTING_METRIC_LABELS[metric]} ({metric})")
print(f"Pairing unit: {unit}")
print(f"Equal-params models included: {analysis['target_labels']}")
print(f"Reference model (for gain summary): {analysis['gain']['reference_label']}")
print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
print(f"Wilcoxon/Holm paired rows used: {int(n_pairs)}")

print("\nEqual-params model means:")
display(model_means.sort_values(ascending=not higher_is_better).to_frame("mean"))

print("\nPairwise Wilcoxon p-values with Holm significance flag:")
p_value_table = wilcoxon_summary["p_value_table"]
display(
    p_value_table.style.format({"p_raw": "{:.6f}"}).apply(
        lambda col: ["background-color: #c7f9cc" if bool(v) else "" for v in col],
        subset=["significant_holm"],
    )
)

print("Average ranks used in the CD diagram (1 = best):")
rank_table = wilcoxon_summary["rank_table"]
display(rank_table.sort_values("average_rank").style.format({"average_rank": "{:.3f}"}))

for fig_key in ["gain_barh", "wilcoxon_cd"]:
    if fig_key in analysis["figures"]:
        fig, _ = analysis["figures"][fig_key]
        display(fig)
        plt.close(fig)

## Transformer Masking Model Comparison

Compare transformer masking variants (from the `transformer_masked` registry family) with paired gains and Wilcoxon-Holm significance.


In [ ]:
import matplotlib.pyplot as plt

from pfns.experiments.model_benchmarks.model_registry import get_models_from_families
from pfns.experiments.model_benchmarks.setting_analysis import (
    SETTING_METRIC_DIRECTION,
    SETTING_METRIC_LABELS,
)
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis

TRANSFORMER_MASK_ANALYSIS = {
    "metric": "accuracy",  # one of: accuracy, roc_auc, log_loss, ece
    "unit": "dataset",  # dataset or split
    "reference_label": None,  # auto-selects best mean metric if None
    "wilcoxon_alpha": 0.05,
}

if all_results is None or all_results.empty:
    raise RuntimeError("No aggregated all_results dataframe is available for transformer mask analysis.")

metric = TRANSFORMER_MASK_ANALYSIS["metric"]
if metric not in SETTING_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SETTING_METRIC_DIRECTION)}")
if metric not in all_results.columns:
    raise RuntimeError(f"Metric column {metric!r} is not present in all_results.")

unit = TRANSFORMER_MASK_ANALYSIS["unit"]
if unit not in {"dataset", "split"}:
    raise ValueError("TRANSFORMER_MASK_ANALYSIS['unit'] must be 'dataset' or 'split'.")

registered_mask_models = list(get_models_from_families(["transformer_masked"]).keys())
available_models = set(all_results["model"].astype(str).unique())
target_models = [model for model in registered_mask_models if model in available_models]

if len(target_models) < 2:
    print(
        "Skipping transformer mask comparison: need at least two transformer_masked models in all_results. "
        f"Found {len(target_models)}: {target_models}."
    )
    print("Tip: set models_to_compare to include get_models_from_families(['transformer_masked']).")
else:
    target_display_models = [format_model_label(model) for model in target_models]
    comparison_results = all_results[all_results["model"].isin(target_models)].copy()
    comparison_results["model"] = comparison_results["model"].map(format_model_label)
    higher_is_better = SETTING_METRIC_DIRECTION[metric]
    model_means = comparison_results.groupby("model", observed=True)[metric].mean().reindex(target_display_models)

    reference_label = TRANSFORMER_MASK_ANALYSIS["reference_label"]
    if reference_label is None:
        reference_label = model_means.idxmax() if higher_is_better else model_means.idxmin()
    elif reference_label not in target_models:
        raise RuntimeError(
            f"Configured reference_label={reference_label!r} is not among available transformer mask models: {target_models}"
        )
    else:
        reference_label = format_model_label(reference_label)

    pair_cols = ["dataset"] if unit == "dataset" else ["dataset", "split"]
    analysis = run_comparison_analysis(
        comparison_results=comparison_results,
        metric_col=metric,
        metric_label=SETTING_METRIC_LABELS[metric],
        compare_col="model",
        target_labels=target_display_models,
        pair_cols=pair_cols,
        higher_better=higher_is_better,
        reference_label=reference_label,
        unit=unit,
        error_bars="ci95",
        comparison_label="model",
        include_boxplot=False,
        include_pairwise_tables=False,
        include_cd_diagram=True,
        wilcoxon_alpha=TRANSFORMER_MASK_ANALYSIS["wilcoxon_alpha"],
        empty_error_message=(
            "No complete paired rows found across transformer mask models. "
            "Try unit='dataset' or evaluate more transformer_masked models."
        ),
    )

    wilcoxon_summary = analysis["wilcoxon"]
    n_pairs = wilcoxon_summary["n_pairs"]

    print(f"Metric: {SETTING_METRIC_LABELS[metric]} ({metric})")
    print(f"Pairing unit: {unit}")
    print(f"Transformer mask models included: {analysis['target_labels']}")
    print(f"Reference model (for gain summary): {analysis['gain']['reference_label']}")
    print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
    print(f"Wilcoxon/Holm paired rows used: {int(n_pairs)}")

    print("\nTransformer mask model means:")
    display(model_means.sort_values(ascending=not higher_is_better).to_frame("mean"))

    print("\nPairwise Wilcoxon p-values with Holm significance flag:")
    p_value_table = wilcoxon_summary["p_value_table"]
    display(
        p_value_table.style.format({"p_raw": "{:.6f}"}).apply(
            lambda col: ["background-color: #c7f9cc" if bool(v) else "" for v in col],
            subset=["significant_holm"],
        )
    )

    print("Average ranks used in the CD diagram (1 = best):")
    rank_table = wilcoxon_summary["rank_table"]
    display(rank_table.sort_values("average_rank").style.format({"average_rank": "{:.3f}"}))

    for fig_key in ["gain_barh", "wilcoxon_cd"]:
        if fig_key in analysis["figures"]:
            fig, _ = analysis["figures"][fig_key]
            display(fig)
            plt.close(fig)

# Model size report for all registry models

In [ ]:
import gc
import os

import pandas as pd
import torch

from pfns.experiments.model_benchmarks.model_registry import get_all_models
from pfns.run_logger import download_model_from_wandb
from pfns.scripts.tabpfn_interface import load_model_workflow

VOCAB_MAPPING_HINTS = ('vocab_mapping', 'vocab_map')

def _is_fla_model(model):
    backbone = getattr(model, 'transformer_layers', None)
    if backbone is None:
        return False
    class_name = backbone.__class__.__name__.lower()
    return class_name == 'flabackbone' or hasattr(backbone, 'fla')

def _is_vocab_mapping_param(name):
    lowered = name.lower()
    return 'fla.embeddings' in lowered

def _load_registry_model(model_config, device):
    base_path = str(model_config.get('base_path') or '.')
    checkpoint_name = str(model_config.get('checkpoint_name') or 'checkpoint.pt')

    wandb_run_id = model_config.get('wandb_run_id')
    if wandb_run_id:
        target_path = download_model_from_wandb(
            wandb_run_id,
            destination_path=model_config.get('destination_path'),
        )
        base_path = os.path.dirname(target_path)
        checkpoint_name = os.path.basename(target_path)

    model, _, _ = load_model_workflow(
        name=checkpoint_name,
        base_path=base_path,
        device=device,
    )
    model.eval()
    return model

def _compute_param_stats(model):
    total_params = 0
    total_bytes = 0
    excluded_params = 0
    excluded_bytes = 0
    excluded_names = []
    is_fla = _is_fla_model(model)

    for name, param in model.named_parameters():
        param_count = int(param.numel())
        param_bytes = param_count * int(param.element_size())
        total_params += param_count
        total_bytes += param_bytes

        if is_fla and _is_vocab_mapping_param(name):
            excluded_params += param_count
            excluded_bytes += param_bytes
            excluded_names.append(name)

    return {
        'is_fla': is_fla,
        'params_total': total_params,
        'params_excluded_vocab_mapping': excluded_params,
        'params_effective': total_params - excluded_params,
        'size_mib_total': total_bytes / (1024 ** 2),
        'size_mib_excluded_vocab_mapping': excluded_bytes / (1024 ** 2),
        'size_mib_effective': (total_bytes - excluded_bytes) / (1024 ** 2),
        'excluded_param_names': ';'.join(excluded_names),
    }

all_models = get_models_from_families(["equal_params"])  # get_all_models()
report_device = globals().get('device', 'cpu')
rows = []

for idx, (model_name, model_config) in enumerate(sorted(all_models.items()), start=1):
    print(f'[{idx}/{len(all_models)}] Loading {model_name}')
    model = None
    row = {
        'model_name': model_name,
        'display_name': model_config.get('display_name', model_name),
        'wandb_run_id': model_config.get('wandb_run_id', ''),
        'status': 'ok',
        'error': '',
    }
    try:
        model = _load_registry_model(model_config, device=report_device)
        row.update(_compute_param_stats(model))
    except Exception as exc:
        row['status'] = 'error'
        row['error'] = f'{type(exc).__name__}: {exc}'
    finally:
        rows.append(row)
        if model is not None:
            del model
        gc.collect()
        if str(report_device).startswith('cuda') and torch.cuda.is_available():
            torch.cuda.empty_cache()

df_model_sizes = pd.DataFrame(rows).sort_values(by=['status', 'model_name']).reset_index(drop=True)
display_cols = [
    'model_name',
    'display_name',
    'is_fla',
    'params_total',
    'params_excluded_vocab_mapping',
    'params_effective',
    'size_mib_total',
    'size_mib_excluded_vocab_mapping',
    'size_mib_effective',
    'status',
    'error',
]
display(df_model_sizes[display_cols])